In [1]:
import cv2
import numpy as np
import json
from PIL import Image

def preprocess_data(image_path, mask_path, pose_json_path, output_dir):
    # 设定最终统一的目标尺寸 (IDM-VTON 通常推荐使用 768x1024 或 384x512)
    # 我们先统一到 768x1024 保证清晰度
    target_w, target_h = 768, 1024

    # --- 1. 处理原始图片 ---
    img = cv2.imread(image_path)
    img = cv2.resize(img, (target_w, target_h))
    cv2.imwrite(f"{output_dir}/image.jpg", img)

    # --- 2. 处理分割掩码 (Parsing Mask) ---
    # 读取你上传的 segmentation_mask.webp
    # 注意：它是单通道灰度图，每个像素值对应一个 ID
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (target_w, target_h), interpolation=cv2.INTER_NEAREST)

    # 创建 Agnostic Mask (生成模型需要的遮罩)
    # 根据你的 Readme: 3(top), 4(dress), 5(skirt), 6(pants), 7(belt) 是要换掉的
    agnostic_mask = np.zeros_like(mask)
    cloth_ids = [3, 4, 5, 6, 7] 
    
    # 我们要把衣服区域涂成白色(255)，代表这块区域模型可以重新生成
    for cid in cloth_ids:
        agnostic_mask[mask == cid] = 255
    
    # 适当加一点点膨胀，防止边缘留白
    kernel = np.ones((5, 5), np.uint8)
    agnostic_mask = cv2.dilate(agnostic_mask, kernel, iterations=1)
    cv2.imwrite(f"{output_dir}/agnostic_mask.png", agnostic_mask)

    # --- 3. 处理姿态数据 (Pose JSON) ---
    with open(pose_json_path, 'r') as f:
        pose_data = json.load(f)
    
    # 创建一张黑底图用来画骨架
    pose_canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)
    
    # 提取关键点并绘制 (以 OpenPose 风格为例)
    # 原 JSON 坐标基于 600x800，需要缩放到 768x1024
    scale_x = target_w / 600
    scale_y = target_h / 800

    keypoints = pose_data['poses'][0]['keypoints']
    for kp in keypoints:
        px, py = int(kp['x'] * scale_x), int(kp['y'] * scale_y)
        if kp['score'] > 0.3: # 只画置信度高的点
            cv2.circle(pose_canvas, (px, py), 5, (0, 255, 0), -1) # 画绿色的点

    cv2.imwrite(f"{output_dir}/pose_skeleton.png", pose_canvas)

    print("预处理完成！生成了 image.jpg, agnostic_mask.png 和 pose_skeleton.png")

# 使用示例
if __name__ == "__main__":
    preprocess_data(
        image_path='input_image.jpg', 
        mask_path='segmentation_mask.webp', 
        pose_json_path='pose_data.json',
        output_dir='./'
    )

error: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'cv::resize'


In [8]:
import cv2
import numpy as np
import json
import os
from PIL import Image  # 导入 Pillow 库来稳健读取图片

def preprocess_data(image_path, mask_path, pose_json_path, output_dir):
    target_w, target_h = 768, 1024

    # --- 1. 使用 Pillow 读取原始图片 ---
    try:
        # Pillow 对 .jpg 和 .webp 的支持比 OpenCV 更稳
        img_pil = Image.open(image_path).convert('RGB')
        img = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
        img = cv2.resize(img, (target_w, target_h))
        save_path = os.path.join(output_dir, "image.jpg")
        success = cv2.imwrite(save_path, img)
        print("保存路径:", save_path)
        print("写入成功:", success)
        print(f"✅ 原始图片已保存至: {save_path}")
    except Exception as e:
        print(f"❌ 原始图片处理失败: {e}")
        return

    # --- 2. 处理分割掩码 ---
    try:
        mask_pil = Image.open(mask_path).convert('L') # 强制转为灰度
        mask = np.array(mask_pil)
        mask = cv2.resize(mask, (target_w, target_h), interpolation=cv2.INTER_NEAREST)

        agnostic_mask = np.zeros_like(mask)
        # 根据你的 Readme，合并衣服相关的 ID
        cloth_ids = [3, 4, 5, 6, 7] 
        for cid in cloth_ids:
            agnostic_mask[mask == cid] = 255
        
        save_path_mask = os.path.join(output_dir, "agnostic_mask.png")
        cv2.imwrite(save_path_mask, agnostic_mask)
        print(f"✅ 掩码已保存至: {save_path_mask}")
    except Exception as e:
        print(f"❌ 掩码处理失败: {e}")
        return

    # --- 3. 处理姿态数据 ---
    try:
        with open(pose_json_path, 'r') as f:
            pose_data = json.load(f)
        
        pose_canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)
        # 你的 JSON 显示原图是 600x800
        scale_x = target_w / 600
        scale_y = target_h / 800

        keypoints = pose_data['poses'][0]['keypoints']
        for kp in keypoints:
            px, py = int(kp['x'] * scale_x), int(kp['y'] * scale_y)
            if kp['score'] > 0.3:
                cv2.circle(pose_canvas, (px, py), 8, (0, 255, 0), -1) 
        save_path_pose = os.path.join(output_dir, "pose_skeleton.png")
        cv2.imwrite(save_path_pose, pose_canvas)
        print(f"✅ 姿态已保存至: {save_path_pose}")
    except Exception as e:
        print(f"❌ JSON处理失败: {e}")

if __name__ == "__main__":
    # 确保这些文件名和你截图里的一模一样
    MY_BASE_PATH = r"D:\大三下课程学习\深度学习\大作业\任务一\任务一\001"
    
    if not os.path.exists(MY_BASE_PATH):
        os.makedirs(MY_BASE_PATH)
        print(f"创建了新目录: {MY_BASE_PATH}")
        
    preprocess_data(
        image_path=os.path.join(MY_BASE_PATH, 'input_image.jpg'), 
        mask_path=os.path.join(MY_BASE_PATH, 'segmentation_mask.webp'), 
        pose_json_path=os.path.join(MY_BASE_PATH, 'pose_data.json'),
        output_dir=MY_BASE_PATH # 结果也保存在这个文件夹
    )

保存路径: D:\大三下课程学习\深度学习\大作业\任务一\任务一\001\image.jpg
写入成功: False
✅ 原始图片已保存至: D:\大三下课程学习\深度学习\大作业\任务一\任务一\001\image.jpg
✅ 掩码已保存至: D:\大三下课程学习\深度学习\大作业\任务一\任务一\001\agnostic_mask.png
✅ 姿态已保存至: D:\大三下课程学习\深度学习\大作业\任务一\任务一\001\pose_skeleton.png


In [6]:
print("当前工作目录:", os.getcwd())
print("保存路径:", os.path.abspath(MY_BASE_PATH))

当前工作目录: D:\大三下课程学习\深度学习\大作业\任务三
保存路径: D:\大三下课程学习\深度学习\大作业\任务一\任务一\001


In [7]:
print("文件列表:", os.listdir(MY_BASE_PATH))

文件列表: ['.ipynb_checkpoints', 'input_image.jpg', 'pose_data.json', 'pose_detection.png', 'segmentation_mask.webp', 'segmentation_overlay.webp', 'Untitled.ipynb']


In [9]:
import cv2
import numpy as np
import json
import os
from PIL import Image


def save_image_safe(img, save_path):
    """
    使用 Pillow 保存图片（支持中文路径）
    """
    try:
        if len(img.shape) == 3:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            Image.fromarray(img).save(save_path)
        else:
            Image.fromarray(img).save(save_path)

        if os.path.exists(save_path):
            print(f"✅ 保存成功: {save_path}")
        else:
            print(f"❌ 保存失败: {save_path}")

    except Exception as e:
        print(f"❌ 图片保存错误: {e}")


def preprocess_data(image_path, mask_path, pose_json_path, output_dir):

    target_w, target_h = 768, 1024

    print("\n========== 开始数据预处理 ==========")

    print("输入图片:", image_path)
    print("输入mask:", mask_path)
    print("输入pose:", pose_json_path)
    print("输出目录:", output_dir)

    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)

    # =============================
    # 1 处理原始图片
    # =============================
    try:

        img_pil = Image.open(image_path).convert("RGB")
        img = np.array(img_pil)

        img = cv2.resize(img, (target_w, target_h))

        save_path = os.path.join(output_dir, "image.jpg")

        save_image_safe(img, save_path)

    except Exception as e:
        print("❌ 原始图片处理失败:", e)
        return

    # =============================
    # 2 处理分割掩码
    # =============================
    try:

        mask_pil = Image.open(mask_path).convert("L")
        mask = np.array(mask_pil)

        mask = cv2.resize(mask, (target_w, target_h), interpolation=cv2.INTER_NEAREST)

        agnostic_mask = np.zeros_like(mask)

        # 根据项目Readme合并衣服区域
        cloth_ids = [3, 4, 5, 6, 7]

        for cid in cloth_ids:
            agnostic_mask[mask == cid] = 255

        save_path_mask = os.path.join(output_dir, "agnostic_mask.png")

        save_image_safe(agnostic_mask, save_path_mask)

    except Exception as e:
        print("❌ 掩码处理失败:", e)
        return

    # =============================
    # 3 处理姿态数据
    # =============================
    try:

        with open(pose_json_path, "r", encoding="utf-8") as f:
            pose_data = json.load(f)

        pose_canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)

        # 原图尺寸
        original_w = 600
        original_h = 800

        scale_x = target_w / original_w
        scale_y = target_h / original_h

        keypoints = pose_data["poses"][0]["keypoints"]

        for kp in keypoints:

            px = int(kp["x"] * scale_x)
            py = int(kp["y"] * scale_y)

            if kp["score"] > 0.3:

                cv2.circle(
                    pose_canvas,
                    (px, py),
                    8,
                    (0, 255, 0),
                    -1
                )

        save_path_pose = os.path.join(output_dir, "pose_skeleton.png")

        save_image_safe(pose_canvas, save_path_pose)

    except Exception as e:
        print("❌ 姿态处理失败:", e)
        return

    print("\n🎉 数据预处理完成")
    print("输出文件列表:", os.listdir(output_dir))


# =============================
# 主程序
# =============================
if __name__ == "__main__":

    BASE_PATH = r"D:\大三下课程学习\深度学习\大作业\任务一\任务一\001"

    preprocess_data(
        image_path=os.path.join(BASE_PATH, "input_image.jpg"),
        mask_path=os.path.join(BASE_PATH, "segmentation_mask.webp"),
        pose_json_path=os.path.join(BASE_PATH, "pose_data.json"),
        output_dir=BASE_PATH
    )


========== 开始数据预处理 ==========
输入图片: D:\大三下课程学习\深度学习\大作业\任务一\任务一\001\input_image.jpg
输入mask: D:\大三下课程学习\深度学习\大作业\任务一\任务一\001\segmentation_mask.webp
输入pose: D:\大三下课程学习\深度学习\大作业\任务一\任务一\001\pose_data.json
输出目录: D:\大三下课程学习\深度学习\大作业\任务一\任务一\001
✅ 保存成功: D:\大三下课程学习\深度学习\大作业\任务一\任务一\001\image.jpg
✅ 保存成功: D:\大三下课程学习\深度学习\大作业\任务一\任务一\001\agnostic_mask.png
✅ 保存成功: D:\大三下课程学习\深度学习\大作业\任务一\任务一\001\pose_skeleton.png

🎉 数据预处理完成
输出文件列表: ['.ipynb_checkpoints', 'agnostic_mask.png', 'image.jpg', 'input_image.jpg', 'pose_data.json', 'pose_detection.png', 'pose_skeleton.png', 'segmentation_mask.webp', 'segmentation_overlay.webp', 'Untitled.ipynb']
